In [1]:
# import azure.ai.agents as agents
# from azure.identity import DefaultAzureCredential
# agents_client = agents.AgentsClient(
#     endpoint=PROJECT_ENDPOINT,
#     credential=DefaultAzureCredential(),
# )
# agent_list = agents_client.list_agents()
# for agent in agent_list:
#     print(f"Agent ID: {agent.id}, Name: {agent.name}") 

In [2]:
import requests
import json
import urllib3
import time
from azure.identity import DefaultAzureCredential
import os
from azure.ai.projects import AIProjectClient
from azure.ai.agents.models import (
    CodeInterpreterTool, 
    MessageTextContent,
    MessageRole,
)
from azure.identity import DefaultAzureCredential
import time


# Disable warnings for unverified HTTPS requests
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

from dotenv import dotenv_values, load_dotenv

config = {
    **dotenv_values(".env")
}

PROJECT_ENDPOINT = config["PROJECT_ENDPOINT"]
WORKFLOW_ID = config["WORKFLOW_ID"]

agent_url = PROJECT_ENDPOINT
workflow_url = f"{PROJECT_ENDPOINT}/workflows"
thread_url =  f"{agent_url}/threads?api-version=2025-05-15-preview"

In [3]:
def get_access_token() -> str:
    credential = DefaultAzureCredential()
    
    # token = credential.get_token("https://management.azure.com/.default")
    token = credential.get_token("https://ai.azure.com/.default")
    print(f"Access token acquired: {token.token[:10]}...")  # Print first 10 chars
    return token.token

def https_post(url: str, data: dict = {}) -> dict:
    token = get_access_token()
    payload = json.dumps(data)
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}",
        "Content-Length": str(len(payload.encode("utf-8")))
    }
    print(f"POST {url}")
    print(f"Payload: {json.dumps(data, indent=2)}")

    response = requests.post(url, headers=headers, data=payload, verify=False)
    print(f"Status Code: {response.status_code}")

    if 200 <= response.status_code < 300:
        try:
            return response.json()
        except json.JSONDecodeError as e:
            print(f"JSON decode error: {e}")
            print(f"Full response text: {response.text}")
            raise Exception(f"Failed to parse JSON response: {e}")
    else:
        headers_str = json.dumps(dict(response.headers))
        raise Exception(f"POST request failed with status code {response.status_code}: {response.text} | Headers: {headers_str}")

def https_get(url: str) -> dict:
    token = get_access_token()
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}"
    }
    print(f"GET {url}")
    response = requests.get(url, headers=headers, verify=False)
    if 200 <= response.status_code < 300:
        return response.json()
    else:
        headers_str = json.dumps(dict(response.headers))
        raise Exception(f"GET request failed with status code {response.status_code}: {response.text} | Headers: {headers_str}")
    
def create_thread(connection: str) -> dict:
    """
    Create a new thread on the server.

    Args:
        connection (str): Base connection URL.

    Returns:
        dict: The thread response with ID.
    """
    url = f"{connection}/threads?api-version=2025-05-15-preview"
    print(f"Creating thread at: {url}")
    return https_post(url, {})

def create_message(connection: str, thread_id: str, message: str, file_ids: list = None, tool_types: list = None) -> dict:
    """
    Create a message on the given thread.

    Args:
        connection (str): Base connection URL.
        thread_id (str): The thread id.
        message (str): The message content.
        file_ids (list, optional): List of file IDs.
        tool_types (list, optional): List of corresponding tool types.

    Returns:
        dict: The API response in JSON.
    """
    file_ids = file_ids or []
    tool_types = tool_types or []
    if len(file_ids) != len(tool_types):
        raise Exception('The length of file_ids and tool_types must be the same.')

    # Construct attachments if available
    attachments = []
    if file_ids:
        for idx, file_id in enumerate(file_ids):
            attachments.append({
                "file_id": file_id,
                "tools": [{"type": tool_types[idx]}]
            })

    # Build payload
    payload = {
        "role": "user",
        "content": message
    }

    if attachments:
        payload["attachments"] = attachments

    url = f"{connection}/threads/{thread_id}/messages?api-version=2025-05-15-preview"
    print(f"Creating message at: {url}")
    return https_post(url, payload)

def create_and_stream_workflow_run(connection: str, thread_id: str, agent_x_id: str):
    """
    Create a workflow run AND handle the streaming response.

    We'll filter the SSE data to show only relevant user-facing messages by checking
    for 'thread.message.delta' events and printing text chunks.

    Additionally, we'll track completed runs/messages to avoid printing duplicates.

    there can be multiple runs for a multi-agent workflow, the run_id of a single run step is not returned.
    """
    url = f"{connection}/threads/{thread_id}/runs?api-version=2025-05-15-preview"
    
    payload = {
        "assistant_id": agent_x_id,  # This should be the actual agent ID you want to use
        "stream": True
    }

    token = get_access_token()
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}"
    }

    print(f"Creating and streaming workflow run at: {url}")
    print(f"Payload: {json.dumps(payload, indent=2)}")

    response = requests.post(url, headers=headers, json=payload, verify=False, stream=True)

    if response.status_code >= 400:
        print(f"ERROR: {response.status_code} - {response.text}")
        raise Exception(f"Workflow run creation failed with status code {response.status_code}: {response.text}")

    # sets to track completed run/message IDs to avoid duplicates

    run_id = None # ""  # there is multiple runs for a multiagent workflow.
    current_event_type = None

    completed_messages = set()
    completed_runs = set()
    message_buffer = ""  # Buffer to accumulate message text from delta events



    print("\n--- Streaming Response ---")

    def print_event_summary(event, data):
        """Print a friendly summary of the event based on its type."""
        if event == "thread.run.created":
            print(f"[Workflow Started] Run created with ID: {data.get('id')}")      
        elif event == "thread.run.in_progress":
            print(f"[Workflow Update] Run {data.get('id')} is in progress.")
        elif event == "thread.run.queued":
            print(f"[Workflow Queued] Run {data.get('id')} is queued.")
        elif event == "thread.run.completed":
            print(f"[Workflow Completed] Run {data.get('id')} completed.")
        elif event == "thread.run.failed":
            print(f"[Workflow Failed] Run {data.get('id')} failed.")
        elif event == "thread.run.step.created":
            print(f"[Step Started] Step {data.get('id')} started. Type: {data.get('type')}")
        elif event == "thread.run.step.in_progress":
            print(f"[Step Update] Step {data.get('id')} is in progress.")
        elif event == "thread.run.step.completed":
            print(f"[Step Completed] Step {data.get('id')} completed.")
        elif event == "thread.message.created":
            print(f"[Message Created] Message {data.get('id')} created.")
        elif event == "thread.message.completed":
            print(f"[Message Completed] Message {data.get('id')} finished.")
        else:
            print(f"[Other Event] {event} (ID: {data.get('id')})")

    try:
        for line in response.iter_lines():
            if not line:
                continue

            line_str = line.decode("utf-8").strip()

            # When encountering a new event, flush any pending message content
            if line_str.startswith("event:"):
                # If previous event was message.delta and we have content, print it
                if current_event_type == "thread.message.delta" and message_buffer:
                    print(f"[Message Content] {message_buffer}")
                    message_buffer = ""
                current_event_type = line_str[6:].strip()
                # (Do not print event header immediately; wait for data)
                continue

            if line_str.startswith("data:"):
                data_content = line_str[5:].strip()
                if data_content == "[DONE]":
                    # Flush any buffered message text
                    if message_buffer:
                        print(f"[Message Content] {message_buffer}")
                    print("\nStream completed [DONE]")
                    break

                try:
                    data = json.loads(data_content)
                except json.JSONDecodeError:
                    continue

                # Special handling for message delta events: accumulate text
                if current_event_type == "thread.message.delta":
                    delta = data.get("delta", {})
                    for piece in delta.get("content", []):
                        if piece.get("type") == "text":
                            message_buffer += piece["text"]["value"]
                    continue  # Skip further processing so the delta text isn’t printed as JSON

                # For message completed events, flush any accumulated text
                if current_event_type == "thread.message.completed":
                    if message_buffer:
                        print(f"[Message Content] {message_buffer}")
                        message_buffer = ""

                # Print a concise, friendly summary for this event
                print_event_summary(current_event_type, data)

                # For message completion, show the final message text
                if current_event_type == "thread.message.completed":
                    for piece in data.get("content", []):
                        if piece.get("type") == "text":
                            print(f"Final Message: {piece['text']['value']}\n")

                # Capture run ID on creation
                if current_event_type == "thread.run.created" and "id" in data:
                    run_id = data["id"]

                # Mark completed events to avoid duplicate prints if needed
                if current_event_type == "thread.run.completed":
                    rid = data.get("id")
                    if rid and rid not in completed_runs:
                        completed_runs.add(rid)
                elif current_event_type == "thread.message.completed":
                    mid = data.get("id")
                    if mid and mid not in completed_messages:
                        completed_messages.add(mid)

    except KeyboardInterrupt:
        print("\nStreaming interrupted by user")
    except Exception as e:
        print(f"Error while streaming: {e}")

    # return run_id

In [4]:
project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

thread = project_client.agents.threads.create()

thread_id = thread.id # Get the thread ID e.g. thread_IJSKkVkcmjNjiXZGWu1U3jiz
print(f"Thread ID: {thread_id}")

Thread ID: thread_RlgXvgaKjdycx18iRRZBb3CD


In [5]:
# project_client.agents._config.credential_scopes
# project_client.agents._config.endpoint
# user_message = "Does this policy cover my eye treatment?"
# message_result = project_client.agents.messages.create(
#     thread_id=thread_id, 
#     role=MessageRole.USER, 
#     content=user_message
# )

In [6]:
user_message = "How do microsoft do quantum computer in 2025?"
message_result = create_message(agent_url, thread_id, user_message)
print(message_result)

Creating message at: https://foundry-agent-fdl-resource.services.ai.azure.com/api/projects/foundry-agent-fdl/threads/thread_RlgXvgaKjdycx18iRRZBb3CD/messages?api-version=2025-05-15-preview
Access token acquired: eyJ0eXAiOi...
POST https://foundry-agent-fdl-resource.services.ai.azure.com/api/projects/foundry-agent-fdl/threads/thread_RlgXvgaKjdycx18iRRZBb3CD/messages?api-version=2025-05-15-preview
Payload: {
  "role": "user",
  "content": "How do microsoft do quantum computer in 2025?"
}
Status Code: 200
{'id': 'msg_as86fBxcvW48HaE0zRgQYI8h', 'object': 'thread.message', 'created_at': 1754574056, 'assistant_id': None, 'thread_id': 'thread_RlgXvgaKjdycx18iRRZBb3CD', 'run_id': None, 'role': 'user', 'content': [{'type': 'text', 'text': {'value': 'How do microsoft do quantum computer in 2025?', 'annotations': []}}], 'attachments': [], 'metadata': {}}


In [7]:
create_and_stream_workflow_run(workflow_url, thread_id, WORKFLOW_ID)

Access token acquired: eyJ0eXAiOi...
Creating and streaming workflow run at: https://foundry-agent-fdl-resource.services.ai.azure.com/api/projects/foundry-agent-fdl/workflows/threads/thread_RlgXvgaKjdycx18iRRZBb3CD/runs?api-version=2025-05-15-preview
Payload: {
  "assistant_id": "wf_agent_Zm3xTPhClQn0opblGRMsMUWd",
  "stream": true
}

--- Streaming Response ---
[Workflow Started] Run created with ID: wf_run_Dz18XQZBoBT89rzeAKtoXr7g
[Workflow Queued] Run wf_run_Dz18XQZBoBT89rzeAKtoXr7g is queued.
[Workflow Update] Run wf_run_Dz18XQZBoBT89rzeAKtoXr7g is in progress.
[Step Started] Step wf_step_DOPAzo9vStqYG1tbmaeRLxUt started. Type: tool_calls
[Workflow Started] Run created with ID: run_oJ36XeDjrQUOqdQnr7AKsf9g
[Workflow Queued] Run run_oJ36XeDjrQUOqdQnr7AKsf9g is queued.
[Workflow Update] Run run_oJ36XeDjrQUOqdQnr7AKsf9g is in progress.
[Step Started] Step step_pHRScDEkS5Z7XDTwU5pOnrri started. Type: tool_calls
[Step Update] Step step_pHRScDEkS5Z7XDTwU5pOnrri is in progress.
[Other Even